# NLP Lab Assignment 8
**AI357 — Natural Language Processing**  
**SVNIT, Surat**

This notebook solves both parts of the assignment:
1. Programs and worked results for all five questions in the supplied midsem paper.
2. Noisy-channel spell correction using the three supplied data files, with sample corrections and accuracy tables in the format of the reference notebook.

Run the cells from top to bottom. The `Spell-Correction` folder must be beside this notebook. Python 3 and pandas are required. All displayed results below are produced by the code cells.


In [1]:
from collections import Counter, defaultdict
from fractions import Fraction
from itertools import combinations
from pathlib import Path
import math
import re
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)


## 1. Midsem programs
### Q1. Noun finite-state transducer

Input is a lexical noun such as `box+N+PL`; output is its surface form, `boxes`.
The singular path copies the stem. The plural path appends `es` after `s`, `x`, `z`, `ch`, or `sh`, and `s` otherwise, exactly as specified.

The transducer tracks the stem ending while copying its letters. `+N` selects the suffix state; `+SG` or `+PL` completes the word. Reversing the input/output relation gives the corresponding morphological analysis. The stem is assumed to be an English noun; no noun dictionary is specified in the question.


In [2]:
states = {
    'START': 'Initial state; a nonempty stem is required',
    'R': 'Stem has an ordinary ending',
    'S': 'Stem ends in s',
    'C': 'Stem ends in c',
    'E': 'Stem ends in x, z, ch, or sh',
    'N_R': '+N consumed; plural takes s',
    'N_E': '+N consumed; plural takes es',
    'FINAL': 'Accepting state',
}
display(pd.DataFrame(states.items(), columns=['State', 'Meaning']))

def letter_target(state, char):
    if char == 's':
        return 'S'
    if char == 'c':
        return 'C'
    if char in 'xz' or (char == 'h' and state in {'S', 'C'}):
        return 'E'
    return 'R'

transitions = {}
alphabet = 'abcdefghijklmnopqrstuvwxyz'
for state in ['START', 'R', 'S', 'C', 'E']:
    for char in alphabet:
        transitions[state, char] = (letter_target(state, char), char)
for state in ['R', 'S', 'C', 'E']:
    transitions[state, '+N'] = ('N_E' if state in {'S', 'E'} else 'N_R', '')
for state, suffix in [('N_R', 's'), ('N_E', 'es')]:
    transitions[state, '+SG'] = ('FINAL', '')
    transitions[state, '+PL'] = ('FINAL', suffix)

# Compact transition table: grouped character arcs still copy the input letter.
rows = []
for state in ['START', 'R', 'S', 'C', 'E']:
    groups = defaultdict(list)
    for char in alphabet:
        groups[transitions[state, char][0]].append(char)
    for target, chars in groups.items():
        rows.append([state, ', '.join(chars), 'same input letter', target])
for (state, symbol), (target, output) in transitions.items():
    if symbol.startswith('+'):
        rows.append([state, symbol, output or 'epsilon', target])
display(pd.DataFrame(rows, columns=['From', 'Input', 'Output', 'To']))

def transduce(lexical_word):
    match = re.fullmatch(r'([a-z]+)(\+N)(\+SG|\+PL)', lexical_word)
    if not match:
        raise ValueError('Use lowercase stem+N+SG or stem+N+PL')
    stem, noun_tag, number_tag = match.groups()
    state, output, path = 'START', [], ['START']
    for symbol in [*stem, noun_tag, number_tag]:
        state, emitted = transitions[state, symbol]
        output.append(emitted)
        path.append(state)
    if state != 'FINAL':
        raise ValueError('Input is not accepted')
    return ''.join(output), ' -> '.join(path)

examples = ['cat', 'bus', 'box', 'waltz', 'church', 'dish']
rows = []
for noun in examples:
    for number in ['SG', 'PL']:
        lexical = f'{noun}+N+{number}'
        surface, path = transduce(lexical)
        rows.append([lexical, surface, path])
display(pd.DataFrame(rows, columns=['Lexical form', 'Surface form', 'State path']))


,State,Meaning
0,START,Initial state; a nonempty stem is required
1,R,Stem has an ordinary ending
2,S,Stem ends in s
3,C,Stem ends in c
4,E,"Stem ends in x, z, ch, or sh"
5,N_R,+N consumed; plural takes s
6,N_E,+N consumed; plural takes es
7,FINAL,Accepting state


,From,Input,Output,To
0,START,"a, b, d, e, f, g, h, i, j, k, l, m, n, o, p, q, r, t, u, v, w, y",same input letter,R
1,START,c,same input letter,C
2,START,s,same input letter,S
3,START,"x, z",same input letter,E
4,R,"a, b, d, e, f, g, h, i, j, k, l, m, n, o, p, q, r, t, u, v, w, y",same input letter,R
5,R,c,same input letter,C
6,R,s,same input letter,S
7,R,"x, z",same input letter,E
8,S,"a, b, d, e, f, g, i, j, k, l, m, n, o, p, q, r, t, u, v, w, y",same input letter,R
9,S,c,same input letter,C


,Lexical form,Surface form,State path
0,cat+N+SG,cat,START -> C -> R -> R -> N_R -> FINAL
1,cat+N+PL,cats,START -> C -> R -> R -> N_R -> FINAL
2,bus+N+SG,bus,START -> R -> R -> S -> N_E -> FINAL
3,bus+N+PL,buses,START -> R -> R -> S -> N_E -> FINAL
4,box+N+SG,box,START -> R -> R -> E -> N_E -> FINAL
5,box+N+PL,boxes,START -> R -> R -> E -> N_E -> FINAL
6,waltz+N+SG,waltz,START -> R -> R -> R -> R -> E -> N_E -> FINAL
7,waltz+N+PL,waltzes,START -> R -> R -> R -> R -> E -> N_E -> FINAL
8,church+N+SG,church,START -> C -> E -> R -> R -> C -> E -> N_E -> FINAL
9,church+N+PL,churches,START -> C -> E -> R -> R -> C -> E -> N_E -> FINAL


### Q2. WordPiece: two merges and tokenization of `newest`

The corpus is exactly the one in the question. Start with character tokens; `##` denotes a character or subword occurring inside a word.
At each merge, use the WordPiece score

$$\operatorname{score}(a,b)=\frac{f(a,b)}{f(a)f(b)}.$$

All frequencies include the corpus word counts. Ties, if any, are resolved lexicographically. After two merges, tokenize `newest` by longest matching vocabulary pieces.


In [3]:
corpus = [
    'low lower low lower lowest',
    'low lowest low lowest',
    'low new',
    'new new newer newer',
    'new',
]
word_counts = Counter(' '.join(corpus).split())
splits = {word: [word[0]] + ['##' + char for char in word[1:]] for word in word_counts}
vocabulary = {piece for parts in splits.values() for piece in parts}
display(pd.DataFrame([[word, count, ' '.join(splits[word])] for word, count in word_counts.items()],
                     columns=['Word', 'Frequency', 'Initial pieces']))

for step in range(1, 3):
    token_counts, pair_counts = Counter(), Counter()
    for word, pieces in splits.items():
        count = word_counts[word]
        for token in pieces:
            token_counts[token] += count
        for pair in zip(pieces, pieces[1:]):
            pair_counts[pair] += count
    scores = {pair: Fraction(count, token_counts[pair[0]] * token_counts[pair[1]])
              for pair, count in pair_counts.items()}
    ranked = sorted(scores, key=lambda pair: (-scores[pair], pair))
    print(f'Merge {step}: pair scores before merging')
    display(pd.DataFrame([[a, b, pair_counts[a, b], token_counts[a], token_counts[b],
                           str(scores[a, b]), float(scores[a, b])] for a, b in ranked],
                         columns=['a', 'b', 'f(a,b)', 'f(a)', 'f(b)', 'Exact score', 'Score']))
    best = ranked[0]
    merged = best[0] + best[1].removeprefix('##')
    vocabulary.add(merged)
    for word, pieces in splits.items():
        new_pieces, i = [], 0
        while i < len(pieces):
            if i + 1 < len(pieces) and tuple(pieces[i:i+2]) == best:
                new_pieces.append(merged)
                i += 2
            else:
                new_pieces.append(pieces[i])
                i += 1
        splits[word] = new_pieces
    print(f'Selected: {best[0]} + {best[1]} -> {merged}')
    display(pd.DataFrame([[word, ' '.join(parts)] for word, parts in splits.items()],
                         columns=['Word', 'Pieces after merge']))

def wordpiece_tokenize(word, vocabulary):
    result, start = [], 0
    while start < len(word):
        for end in range(len(word), start, -1):
            token = ('##' if start else '') + word[start:end]
            if token in vocabulary:
                result.append(token)
                start = end
                break
        else:
            return ['[UNK]']
    return result

print('Tokenization of newest:', wordpiece_tokenize('newest', vocabulary))


,Word,Frequency,Initial pieces
0,low,5,l ##o ##w
1,lower,2,l ##o ##w ##e ##r
2,lowest,3,l ##o ##w ##e ##s ##t
3,new,4,n ##e ##w
4,newer,2,n ##e ##w ##e ##r


Merge 1: pair scores before merging


,a,b,"f(a,b)",f(a),f(b),Exact score,Score
0,##s,##t,3,3,3,1/3,0.333333
1,l,##o,10,10,10,1/10,0.100000
2,##e,##r,4,13,4,1/13,0.076923
3,##e,##s,3,13,3,1/13,0.076923
4,n,##e,6,6,13,1/13,0.076923
5,##o,##w,10,10,16,1/16,0.062500
6,##w,##e,7,16,13,7/208,0.033654
7,##e,##w,6,13,16,3/104,0.028846


Selected: ##s + ##t -> ##st


,Word,Pieces after merge
0,low,l ##o ##w
1,lower,l ##o ##w ##e ##r
2,lowest,l ##o ##w ##e ##st
3,new,n ##e ##w
4,newer,n ##e ##w ##e ##r


Merge 2: pair scores before merging


,a,b,"f(a,b)",f(a),f(b),Exact score,Score
0,l,##o,10,10,10,1/10,0.100000
1,##e,##r,4,13,4,1/13,0.076923
2,##e,##st,3,13,3,1/13,0.076923
3,n,##e,6,6,13,1/13,0.076923
4,##o,##w,10,10,16,1/16,0.062500
5,##w,##e,7,16,13,7/208,0.033654
6,##e,##w,6,13,16,3/104,0.028846


Selected: l + ##o -> lo


,Word,Pieces after merge
0,low,lo ##w
1,lower,lo ##w ##e ##r
2,lowest,lo ##w ##e ##st
3,new,n ##e ##w
4,newer,n ##e ##w ##e ##r


Tokenization of newest: ['n', '##e', '##w', '##e', '##st']


### Q3. TF-IDF and closest sentences

Use raw word counts for TF and unsmoothed $\operatorname{IDF}(w)=\ln(N/\operatorname{df}(w))$, where $N=4$.
Each sentence is a vector of TF-IDF weights. Compare sentence vectors with cosine similarity. No start/end markers are added.

$$\cos(\mathbf{a},\mathbf{b})=\frac{\mathbf{a}\cdot\mathbf{b}}{\|\mathbf{a}\|\|\mathbf{b}\|}.$$


In [4]:
sentences = ['cat dog animal', 'cat cat pet', 'dog pet animal', 'cat dog pet']
documents = [sentence.split() for sentence in sentences]
terms = ['cat', 'dog', 'animal', 'pet']
df = {term: sum(term in document for document in documents) for term in terms}
idf = {term: math.log(len(documents) / df[term]) for term in terms}
display(pd.DataFrame([[term, df[term], f'ln(4/{df[term]})', idf[term]] for term in terms],
                     columns=['Term', 'df', 'IDF calculation', 'IDF']))
tf = [[document.count(term) for term in terms] for document in documents]
vectors = [[count * idf[term] for term, count in zip(terms, counts)] for counts in tf]
print('Raw TF:')
display(pd.DataFrame(tf, columns=terms, index=['S1', 'S2', 'S3', 'S4']))
print('TF-IDF = TF * IDF:')
display(pd.DataFrame(vectors, columns=terms, index=['S1', 'S2', 'S3', 'S4']))
norms = [math.sqrt(sum(value**2 for value in vector)) for vector in vectors]
display(pd.DataFrame([[f'S{i+1}', sentence, norms[i]] for i, sentence in enumerate(sentences)],
                     columns=['Sentence', 'Text', 'Vector norm']))
rows = []
for i, j in combinations(range(4), 2):
    dot = sum(a * b for a, b in zip(vectors[i], vectors[j]))
    denominator = norms[i] * norms[j]
    rows.append([f'S{i+1}', f'S{j+1}', dot, denominator, dot / denominator])
display(pd.DataFrame(rows, columns=['Sentence A', 'Sentence B', 'Dot product', 'Norm product', 'Cosine similarity']))
best = max(rows, key=lambda row: row[-1])
print(f'Closest sentences: {best[0]} and {best[1]} (cosine similarity = {best[-1]:.6f}).')


,Term,df,IDF calculation,IDF
0,cat,3,ln(4/3),0.287682
1,dog,3,ln(4/3),0.287682
2,animal,2,ln(4/2),0.693147
3,pet,3,ln(4/3),0.287682


Raw TF:


,cat,dog,animal,pet
S1,1,1,1,0
S2,2,0,0,1
S3,0,1,1,1
S4,1,1,0,1


TF-IDF = TF * IDF:


,cat,dog,animal,pet
S1,0.287682,0.287682,0.693147,0.000000
S2,0.575364,0.000000,0.000000,0.287682
S3,0.000000,0.287682,0.693147,0.287682
S4,0.287682,0.287682,0.000000,0.287682


,Sentence,Text,Vector norm
0,S1,cat dog animal,0.803726
1,S2,cat cat pet,0.643277
2,S3,dog pet animal,0.803726
3,S4,cat dog pet,0.498280


,Sentence A,Sentence B,Dot product,Norm product,Cosine similarity
0,S1,S2,0.165522,0.517018,0.320147
1,S1,S3,0.563214,0.645975,0.871882
2,S1,S4,0.165522,0.400480,0.413308
3,S2,S3,0.082761,0.517018,0.160074
4,S2,S4,0.248283,0.320532,0.774597
5,S3,S4,0.165522,0.400480,0.413308


Closest sentences: S1 and S3 (cosine similarity = 0.871882).


### Q4. Bernoulli Naive Bayes: spam classification

The three binary features are `free`, `win`, and `offer`. Estimate the priors and conditional probabilities from the seven given emails. No smoothing is specified; use the observed proportions.

For E8 = (0, 0, 0),
$$P(C\mid E8)\propto P(C)\prod_{j=1}^{3}P(X_j=0\mid C).$$


In [5]:
emails = [
    ('E1', 1, 1, 1, 'Spam'), ('E2', 1, 1, 0, 'Spam'),
    ('E3', 1, 0, 1, 'Spam'), ('E4', 0, 1, 1, 'Spam'),
    ('E5', 0, 0, 1, 'NotSpam'), ('E6', 0, 1, 0, 'NotSpam'),
    ('E7', 1, 0, 0, 'NotSpam'),
]
display(pd.DataFrame(emails, columns=['Email', 'free', 'win', 'offer', 'Class']))
feature_names = ['free', 'win', 'offer']
details, summary, scores = [], [], {}
for label in ['Spam', 'NotSpam']:
    group = [row for row in emails if row[-1] == label]
    prior = Fraction(len(group), len(emails))
    likelihood, factors = Fraction(1), []
    for column, feature in enumerate(feature_names, 1):
        present = sum(row[column] for row in group)
        p1 = Fraction(present, len(group))
        p0 = 1 - p1
        details.append([label, feature, present, len(group) - present, str(p1), str(p0)])
        likelihood *= p0
        factors.append(str(p0))
    score = prior * likelihood
    scores[label] = score
    summary.append([label, str(prior), ' * '.join(factors), str(likelihood), str(score), float(score)])
display(pd.DataFrame(details, columns=['Class', 'Feature', 'Count(1)', 'Count(0)', 'P(1|class)', 'P(0|class)']))
display(pd.DataFrame(summary, columns=['Class', 'Prior', 'P(E8|class) calculation', 'Likelihood', 'Prior * likelihood', 'Score']))
total = sum(scores.values())
display(pd.DataFrame([[label, str(score / total), float(score / total)] for label, score in scores.items()],
                     columns=['Class', 'Normalized posterior (exact)', 'P(class|E8)']))
print('Prediction for E8 (0, 0, 0):', max(scores, key=scores.get))


,Email,free,win,offer,Class
0,E1,1,1,1,Spam
1,E2,1,1,0,Spam
2,E3,1,0,1,Spam
3,E4,0,1,1,Spam
4,E5,0,0,1,NotSpam
5,E6,0,1,0,NotSpam
6,E7,1,0,0,NotSpam


,Class,Feature,Count(1),Count(0),P(1|class),P(0|class)
0,Spam,free,3,1,3/4,1/4
1,Spam,win,3,1,3/4,1/4
2,Spam,offer,3,1,3/4,1/4
3,NotSpam,free,1,2,1/3,2/3
4,NotSpam,win,1,2,1/3,2/3
5,NotSpam,offer,1,2,1/3,2/3


,Class,Prior,P(E8|class) calculation,Likelihood,Prior * likelihood,Score
0,Spam,4/7,1/4 * 1/4 * 1/4,1/64,1/112,0.008929
1,NotSpam,3/7,2/3 * 2/3 * 2/3,8/27,8/63,0.126984


,Class,Normalized posterior (exact),P(class|E8)
0,Spam,9/137,0.065693
1,NotSpam,128/137,0.934307


Prediction for E8 (0, 0, 0): NotSpam


### Q5. Kneser–Ney smoothing

Use $d=0.5$ and count bigrams within each of the five sentences, without start/end markers.

$$P_{KN}(w\mid h)=\frac{\max(c(h,w)-d,0)}{c(h)}+
\frac{dN_{1+}(h,*)}{c(h)}\frac{N_{1+}(*,w)}{N_{1+}(*,*)}.$$

Here $N_{1+}(h,*)$ counts distinct followers, $N_{1+}(*,w)$ counts distinct predecessors, and $N_{1+}(*,*)$ counts distinct bigram types. The queried histories `love` and `like` always have a successor.


In [6]:
sentences = ['I like NLP', 'I like AI', 'You like NLP', 'You love AI', 'We love NLP']
bigrams = Counter()
followers, predecessors = defaultdict(set), defaultdict(set)
for sentence in sentences:
    words = sentence.split()
    for history, word in zip(words, words[1:]):
        bigrams[history, word] += 1
        followers[history].add(word)
        predecessors[word].add(history)
display(pd.DataFrame([[h, w, count] for (h, w), count in bigrams.items()],
                     columns=['History', 'Next word', 'Bigram count']))
print('Distinct bigram types:', len(bigrams))
display(pd.DataFrame([[word, ', '.join(sorted(previous)), len(previous), f'{len(previous)}/{len(bigrams)}']
                      for word, previous in predecessors.items()],
                     columns=['Word', 'Distinct predecessors', 'Predecessor count', 'Continuation probability']))
discount = Fraction(1, 2)
for history, word in [('love', 'AI'), ('like', 'NLP')]:
    context_count = sum(count for (h, w), count in bigrams.items() if h == history)
    count = bigrams[history, word]
    n_followers = len(followers[history])
    continuation = Fraction(len(predecessors[word]), len(bigrams))
    direct = max(Fraction(count) - discount, 0) / context_count
    weight = discount * n_followers / context_count
    probability = direct + weight * continuation
    print(f'P_KN({word}|{history}) = ({count} - 0.5)/{context_count}'
          f' + (0.5 * {n_followers}/{context_count}) * {continuation}'
          f' = {probability} = {float(probability):.6f}')


,History,Next word,Bigram count
0,I,like,2
1,like,NLP,2
2,like,AI,1
3,You,like,1
4,You,love,1
5,love,AI,1
6,We,love,1
7,love,NLP,1


Distinct bigram types: 8


,Word,Distinct predecessors,Predecessor count,Continuation probability
0,like,"I, You",2,2/8
1,NLP,"like, love",2,2/8
2,AI,"like, love",2,2/8
3,love,"We, You",2,2/8


P_KN(AI|love) = (1 - 0.5)/2 + (0.5 * 2/2) * 1/4 = 3/8 = 0.375000
P_KN(NLP|like) = (2 - 0.5)/3 + (0.5 * 2/3) * 1/4 = 7/12 = 0.583333


## 2. Spell correction using the noisy channel model

Use `count_unigram.txt` for word frequencies, `count_1edit.txt` for edit counts, and `spell-errors.txt` for error/correction pairs.

$$\hat w=\arg\max_{w\in\mathrm{Candidates}(x)} P(w)P(x\mid w),\qquad
P(w)=\frac{\operatorname{count}(w)}{\sum_v\operatorname{count}(v)}.$$

This section reproduces the supplied notebook's candidate and scoring procedure and its result-table format. It adds known corrections from `spell-errors.txt` to the dictionary candidates for each input. **The reported accuracy is therefore reference-assisted, in-sample accuracy: the evaluated correct answers help supply candidates. It does not measure performance on unseen spelling errors.**

The reference uses `edit_count + 1` as a channel score, rather than a fully normalized conditional probability. Its fallback scores are 1.0 or 0.1. We retain those scoring rules to make the results comparable. Each comma-separated error is evaluated once; a suffix such as `*5` is stripped, as in the reference notebook. All code below is executed against the local files.


In [7]:
candidate_dirs = [Path('Spell-Correction'), Path('Lab-Assignment-8') / 'Spell-Correction']
data_dir = next((path for path in candidate_dirs if path.is_dir()), None)
if data_dir is None:
    raise FileNotFoundError('Run from Lab-Assignment-8 or the NLP-LAB repository root.')
print('Data directory:', data_dir.resolve())


Data directory: D:\SEM5\NLP\NLP-LAB\Lab-Assignment-8\Spell-Correction


In [8]:
# Step 1: Load the three provided files
def load_unigram_counts(file_path):
    word_counts = {}
    with open(file_path, "r", encoding="latin-1") as f:
        for line in f:
            parts = line.strip().split("\t")
            if len(parts) == 2:
                try:
                    word_counts[parts[0].lower()] = int(parts[1])
                except ValueError:
                    pass
    return word_counts

def load_edit_counts(file_path):
    edit_counts = {}
    with open(file_path, "r", encoding="latin-1") as f:
        for line in f:
            parts = line.strip().split("\t")
            if len(parts) == 2:
                try:
                    edit_counts[parts[0].lower()] = int(parts[1])
                except ValueError:
                    pass
    return edit_counts

def load_spell_errors(file_path):
    test_pairs = []
    with open(file_path, "r", encoding="latin-1") as f:
        for line in f:
            line = line.strip()
            if ":" not in line:
                continue
            correct_word, misspellings = line.split(":", 1)
            correct_word = correct_word.strip().lower()
            for error in misspellings.split(","):
                error = error.strip().lower()
                # Remove optional frequency multiplier like 'fore*5'
                error = re.sub(r"\*\d+$", "", error)
                if error:
                    test_pairs.append((error, correct_word))
    return test_pairs

unigram_counts = load_unigram_counts(data_dir / "count_unigram.txt")
edit_counts = load_edit_counts(data_dir / "count_1edit.txt")
error_pairs = load_spell_errors(data_dir / "spell-errors.txt")

df_summary = pd.DataFrame([
    {"File Name": "count_unigram.txt", "Description": "Vocabulary word frequencies", "Count / Entries": len(unigram_counts)},
    {"File Name": "count_1edit.txt", "Description": "Character confusion error counts", "Count / Entries": len(edit_counts)},
    {"File Name": "spell-errors.txt", "Description": "Test error-to-correct word pairs", "Count / Entries": len(error_pairs)}
])
print("=== Dataset Summary ===")
display(df_summary)


,File Name,Description,Count / Entries
0,count_unigram.txt,Vocabulary word frequencies,333333
1,count_1edit.txt,Character confusion error counts,1542
2,spell-errors.txt,Test error-to-correct word pairs,39710


=== Dataset Summary ===


### Candidate generation and edit scores

Generate single-edit dictionary candidates through deletion, insertion, substitution, and adjacent transposition. Include the reference's known corrections, which can also be more than one edit away. Identify edits as `correct substring|observed substring`, following the reference's convention. At word start, its boundary symbol is `>`.


In [9]:
# Step 2: Candidate Generation and Edit Identification
# Map known errors to correct words from the dataset
known_candidates = defaultdict(set)
for error, correct in error_pairs:
    known_candidates[error].add(correct)

def generate_edit_distance_one_words(word):
    """Generates all strings at edit distance 1 from word."""
    letters = "abcdefghijklmnopqrstuvwxyz"
    splits = [(word[:i], word[i:]) for i in range(len(word) + 1)]

    deletes = [left + right[1:] for left, right in splits if right]
    transposes = [left + right[1] + right[0] + right[2:] for left, right in splits if len(right) > 1]
    replaces = [left + c + right[1:] for left, right in splits if right for c in letters]
    inserts = [left + c + right for left, right in splits for c in letters]

    return set(deletes + transposes + replaces + inserts)

def get_candidates(error_word):
    """Return candidates: known correct words + unigram words at edit distance 1."""
    candidates = set(known_candidates.get(error_word, set()))
    for candidate in generate_edit_distance_one_words(error_word):
        if candidate in unigram_counts:
            candidates.add(candidate)
    return candidates

def get_edit_key(cand, err):
    """
    Identifies the exact edit operation key in count_1edit.txt:
    cand is the correct word, err is the misspelled word.
    """
    if cand == err:
        return None
    # 1. Substitution or Transposition
    if len(cand) == len(err):
        diffs = [(i, cand[i], err[i]) for i in range(len(cand)) if cand[i] != err[i]]
        if len(diffs) == 1:
            i, c, e = diffs[0]
            return f"{c}|{e}"
        if len(diffs) == 2:
            i1, c1, e1 = diffs[0]
            i2, c2, e2 = diffs[1]
            if i2 == i1 + 1 and c1 == e2 and c2 == e1:
                return f"{c1}{c2}|{e1}{e2}"
    # 2. Deletion from cand (err is missing a character)
    elif len(cand) == len(err) + 1:
        for i in range(len(cand)):
            if cand[:i] + cand[i+1:] == err:
                del_char = cand[i]
                prev_char = cand[i-1] if i > 0 else ">"
                if prev_char == ">":
                    return f">{del_char}|>"
                else:
                    return f"{prev_char}{del_char}|{prev_char}"
    # 3. Insertion into cand (err has an extra character)
    elif len(cand) + 1 == len(err):
        for i in range(len(err)):
            if err[:i] + err[i+1:] == cand:
                ins_char = err[i]
                prev_char = cand[i-1] if i > 0 else ">"
                if prev_char == ">":
                    return f">|>{ins_char}"
                else:
                    return f"{prev_char}|{prev_char}{ins_char}"
    return None

def compute_channel_likelihood(cand, err):
    """Computes P(err | cand) using count_1edit table."""
    edit_key = get_edit_key(cand, err)
    if edit_key and edit_key in edit_counts:
        return edit_counts[edit_key] + 1.0  # +1 smoothing
    if abs(len(cand) - len(err)) <= 1:
        return 1.0  # fallback smoothing for 1 edit
    return 0.1

print("Candidate and channel likelihood functions ready.")


Candidate and channel likelihood functions ready.


### Sample corrections

Rank candidates using the sum of log prior counts and log channel scores. The common unigram denominator cancels in the ranking. Candidates with equal scores are resolved lexicographically so reruns are deterministic.


In [10]:
# Step 3: Noisy Channel Spell Checker
def correct_word(error_word):
    """
    Finds the most probable correct word using noisy channel scoring:
    argmax_c P(c) * P(err | c)
    """
    error_word = error_word.lower().strip()
    candidates = get_candidates(error_word)
    if not candidates:
        return error_word

    best_candidate = None
    best_log_score = -1e18

    for cand in sorted(candidates):
        prior_count = unigram_counts.get(cand, 1)
        channel_score = compute_channel_likelihood(cand, error_word)

        # log P(c) + log P(x | c)
        score = math.log(prior_count) + math.log(channel_score)

        if score > best_log_score:
            best_log_score = score
            best_candidate = cand

    return best_candidate

# Test sample words specified in the problem statement
sample_errors = [
    "rainning",
    "raning",
    "writtings",
    "disparingly",
    "yello",
    "elegable",
    "accommodat",
    "recieve"
]

sample_results = []
for error in sample_errors:
    correction = correct_word(error)
    sample_results.append({
        "Input Misspelling": error,
        "Corrected Word": correction
    })

df_samples = pd.DataFrame(sample_results)
print("=== Sample Spell Corrections ===")
display(df_samples)


,Input Misspelling,Corrected Word
0,rainning,raining
1,raning,rating
2,writtings,writings
3,disparingly,disparagingly
4,yello,yellow
5,elegable,eligible
6,accommodat,accommodate
7,recieve,receive


=== Sample Spell Corrections ===


### Accuracy and example results

Accuracy is the number of exact matches to the supplied correction divided by the number of listed error/correction pairs. The tables report the actual run, followed by ten correct and ten incorrect predictions.

In [11]:
# Step 4: Evaluate Spell Checker Accuracy on the dataset
print(f"Evaluating spell checker on {len(error_pairs)} test pairs...")

correct_predictions = 0
total_pairs = len(error_pairs)
sample_mistakes = []
sample_correct = []

for idx, (error_word, gold_word) in enumerate(error_pairs):
    predicted_word = correct_word(error_word)
    if predicted_word == gold_word:
        correct_predictions += 1
        if len(sample_correct) < 10:
            sample_correct.append((error_word, gold_word, predicted_word))
    else:
        if len(sample_mistakes) < 10:
            sample_mistakes.append((error_word, gold_word, predicted_word))

accuracy_pct = (correct_predictions / total_pairs) * 100

df_eval = pd.DataFrame([{
    "Total Test Pairs": total_pairs,
    "Correct Predictions": correct_predictions,
    "Incorrect Predictions": total_pairs - correct_predictions,
    "Overall Accuracy (%)": f"{accuracy_pct:.2f}%"
}])

print("=== Reference-assisted Evaluation Results ===")
display(df_eval)

print("\nSample Correct Predictions:")
df_correct_samples = pd.DataFrame(sample_correct, columns=["Error Word", "Expected Word", "Predicted Word"])
display(df_correct_samples)

print("\nSample Incorrect Predictions:")
df_mistake_samples = pd.DataFrame(sample_mistakes, columns=["Error Word", "Expected Word", "Predicted Word"])
display(df_mistake_samples)


,Total Test Pairs,Correct Predictions,Incorrect Predictions,Overall Accuracy (%)
0,39710,26557,13153,66.88%


,Error Word,Expected Word,Predicted Word
0,rainning,raining,raining
1,writtings,writings,writings
2,disparingly,disparagingly,disparagingly
3,yello,yellow,yellow
4,fours,four,four
5,fuore,four,four
6,agression,aggression,aggression
7,loking,looking,looking
8,looing,looking,looking
9,lookin,looking,looking


,Error Word,Expected Word,Predicted Word
0,raning,raining,rating
1,forer,four,fore
2,fore,four,for
3,for,four,or
4,woodes,woods,wooden
5,haing,hanging,hang
6,begining,looking,beginning
7,luing,looking,lung
8,look,looking,took
9,locking,looking,lacking


Evaluating spell checker on 39710 test pairs...
=== Reference-assisted Evaluation Results ===

Sample Correct Predictions:

Sample Incorrect Predictions:


In [12]:
print(f'Reference-assisted accuracy: {correct_predictions:,}/{total_pairs:,} = {accuracy_pct:.2f}%')
print('This uses known corrections as candidates, as in the supplied notebook; it is not held-out accuracy.')

Reference-assisted accuracy: 26,557/39,710 = 66.88%
This uses known corrections as candidates, as in the supplied notebook; it is not held-out accuracy.
